# Offline training from a Hub dataset

Load the dataset pushed by `01_collect_dataset.ipynb`, train a small DQN-style MOUSE model, and push the trained checkpoint to the Hugging Face Hub.

The model is built from three pieces: `StepEmbedder`, `LlamaBackbone`, and a DQN head. `LlamaBackbone` owns all pretrained config and weight loading, so the rest of the notebook reads `backbone.hidden_dim` from the constructed object.

In [ ]:
import torch
from datasets import load_dataset

from mouse_core.data import DataLoader, Datastore
from mouse_core.objectives import DqnObjectiveConfig, dqn_objective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import LlamaBackbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo written by 01_collect_dataset.ipynb.
DATASET_ID = "mouse-example-dataset"

# Dataset subset/config to load from that repo.
DATASET_CONFIG = "frozenlake"

# Hub model repo where the trained checkpoint is uploaded.
MODEL_ID = "mouse-example-model"

# FrozenLake action count; used for the action embedding and DQN head width.
MAX_ACTIONS = 4

# Number of optimizer updates in this small example run.
TRAIN_STEPS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load the collected dataset

Load the dataset pushed by `01_collect_dataset.ipynb` and hand it to `Datastore` for sequential training batches.

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split="train")

store = Datastore()
store.from_dataset(dataset)

print(store)

Datastore(steps=200)


## Build model

This cell downloads `meta-llama/Llama-3.2-1B` unless it is already cached locally. Use a repo you can access, or pass `load_weights=False` to `LlamaBackbone` when you only want the architecture.

In [ ]:
# One call owns pretrained Llama config extraction and weight loading.
backbone = LlamaBackbone(
    pretrained="meta-llama/Llama-3.2-1B",
    num_layers=2,
)
hidden_dim = backbone.hidden_dim

# Build the encoder from the backbone's public metadata.
encoder = StepEmbedder(
    hidden_dim=hidden_dim,
    modalities=[
        {"name": "action", "embed": "discrete", "vocab_size": MAX_ACTIONS, "std": 0.02, "tokens": 1, "include_type_token": True},
        {"name": "reward", "embed": "rff", "std": 0.02, "in_min": 0.01, "in_max": 10.0, "tokens": 1, "include_type_token": True},
        {"name": "done", "embed": "discrete", "vocab_size": 3, "std": 0.02, "tokens": 1, "include_type_token": True},
        {"name": "scratch", "embed": "learnable", "tokens": 1, "include_type_token": True},
    ],
    concat_modalities=False,
)

heads = DiscreteActionValueHead(
    in_features=hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=hidden_dim,
    num_layers=1,
    scale=0.01,
)

# Compose the three independent pieces.
model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=heads,
).train().to(device)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
dqn_cfg = DqnObjectiveConfig(weight=1.0, gamma=0.99, tau=0.005)

loader = DataLoader(
    store,
    sequence_length=8,
    batch_size=4,
    sampling="random",
    prefetch=2,
    num_workers=0,
)

for step in range(TRAIN_STEPS):
    step_stream = loader.next_batch().to(device)
    out, _ = model(step_stream)
    loss, metrics = dqn_objective(step_stream, out, dqn_cfg)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    model.polyak_update(action_value_tau=dqn_cfg.tau)
    if step % 10 == 0:
        print(f"step={step} loss={metrics['action_value']:.4f}")

loader.close()
print("Training finished.")

step=0 loss=1.6508
step=10 loss=1.3657
step=20 loss=1.3527
step=30 loss=1.2248
step=40 loss=1.0840
Training finished.


## Save to the Hugging Face Hub

Push the trained checkpoint under the hard-coded model name. The Hub client creates or updates the model repo under your authenticated Hugging Face account.

In [ ]:
model.eval().to("cpu")
hub_url = push_model_to_hub(
    model,
    MODEL_ID,
    private=False,
)
print(f"Uploaded model to {hub_url}")